# PCA sur MNIST

Ce notebook conduit les expérimentations PCA selon trois axes : **visualisation**, **compression/décompression** et **génération de données synthétiques**. Chaque section suit le schéma *Hypothèse, Protocole, Résultat, Conclusion*.

## 0. Setup expérimental

**Reproductibilité** : seed fixe (42), `shuffle=False` sur le DataLoader, split train/test 80/20 déterministe.

**Prétraitement** : centrage seul (moyenne 0) sans standardisation. Sur MNIST (pixels dans [0, 1]), toutes les features sont déjà à la même échelle ; la standardisation par variance modifierait l'interprétation des composantes principales (on préserve la structure de covariance brute).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname("__file__"), "..")))

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import json

from src.pca import PCA
from src.dataset import load_mnist_dataset
from src.helper import extract_full_dataset
from src.metrics import compression_report, Latent

In [ ]:
SEED = 42
rng = np.random.default_rng(SEED)

# Chargement déterministe
dataloader = load_mnist_dataset(train=True, batch_size=4096, download=True, shuffle=False)
images, digit_labels = extract_full_dataset(dataloader)

# Sous-échantillonnage pour l'exploration
N = 10000
X_all = images[:N].flatten(start_dim=1).numpy()
labels_all = digit_labels[:N].numpy()

# Split train/test 80/20 déterministe
indices = rng.permutation(N)
split = int(0.8 * N)
train_idx, test_idx = indices[:split], indices[split:]

X_train, X_test = X_all[train_idx], X_all[test_idx]
y_train, y_test = labels_all[train_idx], labels_all[test_idx]

print(f"X_train : {X_train.shape} | X_test : {X_test.shape}")
print(f"Pixel range : [{X_train.min():.2f}, {X_train.max():.2f}]")
print(f"Prétraitement : centrage seul | Seed : {SEED}")

## 1.1 Analyse de la variance expliquée

**Hypothèse** : La majorité de la variance de MNIST est concentrée dans un petit nombre de composantes, car les images de chiffres vivent dans un sous-espace de faible dimension effective par rapport aux 784 pixels.

**Protocole** : Fit PCA complet (k=784), scree plot + variance cumulée, identification des seuils 90/95/99%.

In [ ]:
pca_full = PCA(n_components=X_train.shape[1])
pca_full.fit(X_train)

total_var = np.sum(pca_full.all_eigenvalues_)
var_ratio = pca_full.all_eigenvalues_ / total_var
cumulative = np.cumsum(var_ratio)

# Seuils
n90 = int(np.searchsorted(cumulative, 0.90) + 1)
n95 = int(np.searchsorted(cumulative, 0.95) + 1)
n99 = int(np.searchsorted(cumulative, 0.99) + 1)

fig = make_subplots(rows=1, cols=2, subplot_titles=["Scree Plot (50 premières)", "Variance cumulée"])

n_show = 50
fig.add_trace(
    go.Bar(x=list(range(1, n_show + 1)), y=var_ratio[:n_show].tolist(), name="Par composante"),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=list(range(1, len(cumulative) + 1)), y=cumulative.tolist(), name="Cumulée", mode="lines"),
    row=1, col=2
)

for pct, n_comp, color in [(0.90, n90, "green"), (0.95, n95, "orange"), (0.99, n99, "red")]:
    fig.add_hline(y=pct, line_dash="dash", line_color=color,
                  annotation_text=f"{int(pct * 100)}% ({n_comp} comp.)", row=1, col=2)

fig.update_layout(height=400, width=1000, title_text="Analyse de la variance expliquée", showlegend=False)
fig.show()

print(f"90% de variance : {n90} composantes")
print(f"95% de variance : {n95} composantes")
print(f"99% de variance : {n99} composantes")

## 1.2 Projection 2D et 3D

**Hypothèse** : Avec seulement 2 ou 3 composantes, les classes de chiffres ne sont pas linéairement séparables. On s'attend à un fort chevauchement entre classes visuellement similaires (4/9, 3/5/8), car la PCA maximise la variance globale, pas la discrimination inter-classe.

**Protocole** : Scatter plot 2D (PC1 vs PC2) et 3D (PC1/PC2/PC3), coloré par label, évalué sur le test set.

In [ ]:
pca_2d = PCA(n_components=2)
pca_2d.fit(X_train)
latent_2d = pca_2d.encode(X_test)

var_2d = float(np.sum(pca_2d.explained_variance) / total_var * 100)

fig = px.scatter(
    x=latent_2d.array[:, 0], y=latent_2d.array[:, 1],
    color=y_test.astype(str),
    color_discrete_sequence=px.colors.qualitative.Set3,
    labels={"x": "PC1", "y": "PC2", "color": "Digit"},
    title=f"Projection PCA 2D (variance expliquée : {var_2d:.1f}%)",
    opacity=0.6
)
fig.update_layout(width=800, height=600)
fig.show()

In [ ]:
pca_3d = PCA(n_components=3)
pca_3d.fit(X_train)
latent_3d = pca_3d.encode(X_test)

var_3d = float(np.sum(pca_3d.explained_variance) / total_var * 100)

fig = px.scatter_3d(
    x=latent_3d.array[:, 0], y=latent_3d.array[:, 1], z=latent_3d.array[:, 2],
    color=y_test.astype(str),
    color_discrete_sequence=px.colors.qualitative.Set3,
    labels={"x": "PC1", "y": "PC2", "z": "PC3", "color": "Digit"},
    title=f"Projection PCA 3D (variance expliquée : {var_3d:.1f}%)",
    opacity=0.5
)
fig.update_layout(width=900, height=700)
fig.show()

**Conclusion (1.2)** : *A compléter après exécution.* Observer le degré de chevauchement entre classes. Les paires 4/9 et 3/5/8 devraient se superposer fortement, confirmant que la PCA (transformation linéaire) ne sépare pas les classes de chiffres dans un espace de si faible dimension. Une méthode non linéaire (t-SNE, UMAP) donnerait des clusters mieux définis, mais sort du cadre de cette étude.

## 1.3 Eigendigits : interprétation des composantes

**Hypothèse** : Les premières composantes capturent des traits globaux (contraste global, épaisseur du trait), tandis que les composantes suivantes encodent des détails locaux (boucles, sérifs, inclinaison) spécifiques à des sous-classes de chiffres.

**Protocole** : Reshape des 10 premiers vecteurs propres en images 28x28.

In [ ]:
n_eigen = 10
fig = make_subplots(rows=2, cols=5, subplot_titles=[f"PC{i + 1}" for i in range(n_eigen)])

for i in range(n_eigen):
    row = i // 5 + 1
    col = i % 5 + 1
    component = pca_full.components[:, i].reshape(28, 28)
    fig.add_trace(
        go.Heatmap(z=component[::-1], colorscale="RdBu_r", showscale=False),
        row=row, col=col
    )
    fig.update_xaxes(visible=False, row=row, col=col)
    fig.update_yaxes(visible=False, row=row, col=col)

fig.update_layout(height=400, width=900, title_text="Eigendigits : les 10 premières composantes principales")
fig.show()

**Conclusion (1.3)** : *A compléter après exécution.* Vérifier si PC1 capture bien un mode de variation global (ex. contraste, taille apparente du chiffre) et si les composantes suivantes correspondent à des traits de plus en plus localisés.

## 2. Compression et courbe Rate-Distortion

C'est l'axe quantitatif central. La courbe rate-distortion (taille totale vs MSE) permettra la comparaison directe avec K-Means, SOM et l'AutoEncoder.

**Hypothèse** : La relation entre variance expliquée cumulée et MSE de reconstruction est quasi-linéaire (la PCA minimise l'erreur quadratique par construction).

**Protocole** : Boucle sur `n_components`, évaluation sur le **test set** via `compression_report`.

In [ ]:
n_components_list = [1, 2, 5, 10, 20, 50, 100, 150, 200, 784]
results = []

for n_comp in n_components_list:
    pca = PCA(n_components=n_comp)
    pca.fit(X_train)

    latent = pca.encode(X_test)
    X_rec = pca.decode(latent)
    codebook = pca.get_codebook()

    report = compression_report(codebook, latent, X_test, X_rec)
    report["n_components"] = n_comp

    # Variance expliquée cumulée pour cette valeur de k
    var_explained = float(np.sum(pca.explained_variance) / total_var * 100)
    report["variance_explained_pct"] = var_explained

    results.append(report)
    print(f"k={n_comp:>3d} | MSE={report['reconstruction_mse']:.6f} | "
          f"Var={var_explained:>5.1f}% | "
          f"Bytes={report['total_compressed_bytes']:>10d} | "
          f"Ratio={report['compression_ratio']:.2f}x")

In [ ]:
total_bytes = [r["total_compressed_bytes"] for r in results]
mse_values = [r["reconstruction_mse"] for r in results]
n_comp_labels = [str(r["n_components"]) for r in results]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=total_bytes, y=mse_values,
    mode="lines+markers+text",
    text=n_comp_labels,
    textposition="top center",
    name="PCA",
    marker=dict(size=8)
))
fig.update_layout(
    title="Courbe Rate-Distortion PCA (test set)",
    xaxis_title="Taille totale (octets) = Codebook + Latents",
    yaxis_title="MSE de reconstruction",
    xaxis_type="log",
    width=900, height=500
)
fig.show()

In [ ]:
var_pcts = [r["variance_explained_pct"] for r in results]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=var_pcts, y=mse_values,
    mode="lines+markers+text",
    text=n_comp_labels, textposition="top right"
))
fig.update_layout(
    title="Vérification : MSE vs Variance expliquée cumulée",
    xaxis_title="Variance expliquée (%)",
    yaxis_title="MSE de reconstruction",
    width=800, height=500
)
fig.show()

**Conclusion (2)** : *A compléter après exécution.* Vérifier que la relation MSE-variance est bien quasi-linéaire. Identifier le "sweet spot" : le k minimal qui donne une reconstruction visuellement acceptable (souvent autour de 90-95% de variance).

## 2.1 Reconstruction progressive : impact visuel du nombre de composantes

Pour donner une intuition concrète de ce que signifie "MSE = X", on reconstruit la même image avec différents k.

In [ ]:
sample_indices = [0, 1, 2, 3, 4]
k_values_vis = [2, 10, 50, 100, 200]

n_rows = len(sample_indices)
n_cols = len(k_values_vis) + 1  # +1 pour l'original

# Pré-entraîner un modèle par k
pca_models = {}
for k in k_values_vis:
    pca_k = PCA(n_components=k)
    pca_k.fit(X_train)
    pca_models[k] = pca_k

fig = make_subplots(rows=n_rows, cols=n_cols,
                    column_titles=["Original"] + [f"k={k}" for k in k_values_vis],
                    horizontal_spacing=0.01, vertical_spacing=0.01)

for row_i, s_idx in enumerate(sample_indices):
    # Image originale
    fig.add_trace(
        go.Heatmap(z=X_test[s_idx].reshape(28, 28)[::-1], colorscale="gray", showscale=False),
        row=row_i + 1, col=1
    )
    for col_j, k in enumerate(k_values_vis):
        latent_k = pca_models[k].encode(X_test[s_idx:s_idx + 1])
        rec = pca_models[k].decode(latent_k)
        fig.add_trace(
            go.Heatmap(z=rec.reshape(28, 28)[::-1], colorscale="gray", showscale=False),
            row=row_i + 1, col=col_j + 2
        )

for r in range(1, n_rows + 1):
    for c in range(1, n_cols + 1):
        fig.update_xaxes(visible=False, row=r, col=c)
        fig.update_yaxes(visible=False, row=r, col=c)

fig.update_layout(height=120 * n_rows, width=120 * n_cols,
                  title_text="Reconstruction progressive (mêmes images, k croissant)")
fig.show()

## 3. Génération de données synthétiques

L'axe le plus délicat pour la PCA. A traiter avec un regard critique.

**Hypothèse** : La PCA est un mauvais générateur car (a) elle ne modélise pas la vraie distribution latente (seulement une approximation gaussienne), et (b) le décodeur est purement linéaire. On s'attend à des images floues et peu réalistes.

### 3.1 Protocole A : échantillonnage gaussien naïf

Modéliser l'espace latent par une gaussienne multivariée (moyenne/covariance estimées sur les z d'entraînement), échantillonner, décoder.

In [ ]:
k_gen = 50
pca_gen = PCA(n_components=k_gen)
pca_gen.fit(X_train)

# Estimer la distribution latente du train set
latent_train = pca_gen.encode(X_train)
z_train = latent_train.array

z_mean = np.mean(z_train, axis=0)
z_cov = np.cov(z_train, rowvar=False)

rng_gen = np.random.default_rng(SEED)
z_samples = rng_gen.multivariate_normal(z_mean, z_cov, size=16).astype(np.float32)
generated = np.clip(pca_gen.decode(Latent(array=z_samples, nature="continuous")), 0, 1)

fig = make_subplots(rows=4, cols=4, subplot_titles=[f"#{i + 1}" for i in range(16)])
for i in range(16):
    row, col = i // 4 + 1, i % 4 + 1
    fig.add_trace(
        go.Heatmap(z=generated[i].reshape(28, 28)[::-1], colorscale="gray", showscale=False),
        row=row, col=col
    )
    fig.update_xaxes(visible=False, row=row, col=col)
    fig.update_yaxes(visible=False, row=row, col=col)

fig.update_layout(height=500, width=500, title_text=f"Génération gaussienne naïve (k={k_gen})")
fig.show()

### 3.2 Protocole B : interpolation latente

Interpolation linéaire entre deux z réels dans l'espace latent. On observe si la transition entre deux chiffres est sémantiquement cohérente.

In [ ]:
# Choisir deux images de classes différentes (3 et 8)
idx_a = np.where(y_test == 3)[0][0]
idx_b = np.where(y_test == 8)[0][0]

z_a = pca_gen.encode(X_test[idx_a:idx_a + 1]).array[0]
z_b = pca_gen.encode(X_test[idx_b:idx_b + 1]).array[0]

n_steps = 10
alphas_interp = np.linspace(0, 1, n_steps)
interpolated = np.array([z_a * (1 - a) + z_b * a for a in alphas_interp], dtype=np.float32)
decoded_interp = np.clip(pca_gen.decode(Latent(array=interpolated, nature="continuous")), 0, 1)

fig = make_subplots(rows=1, cols=n_steps, subplot_titles=[f"t={a:.1f}" for a in alphas_interp])
for i in range(n_steps):
    fig.add_trace(
        go.Heatmap(z=decoded_interp[i].reshape(28, 28)[::-1], colorscale="gray", showscale=False),
        row=1, col=i + 1
    )
    fig.update_xaxes(visible=False, row=1, col=i + 1)
    fig.update_yaxes(visible=False, row=1, col=i + 1)

fig.update_layout(height=200, width=1200,
                  title_text=f"Interpolation latente : {y_test[idx_a]} vers {y_test[idx_b]} (k={k_gen})")
fig.show()

### 3.3 Qualité de génération en fonction de k

Trop peu de composantes = perte de détail. Beaucoup = espace latent haute dimension où l'hypothèse gaussienne est moins fiable.

In [ ]:
k_values_gen = [5, 10, 20, 50, 100, 200]

fig = make_subplots(rows=len(k_values_gen), cols=8,
                    row_titles=[f"k={k}" for k in k_values_gen],
                    horizontal_spacing=0.01, vertical_spacing=0.02)

rng_gen2 = np.random.default_rng(SEED)
for row_i, k in enumerate(k_values_gen):
    pca_g = PCA(n_components=k)
    pca_g.fit(X_train)
    z_tr = pca_g.encode(X_train).array
    z_m = np.mean(z_tr, axis=0)
    z_c = np.cov(z_tr, rowvar=False)
    z_s = rng_gen2.multivariate_normal(z_m, z_c, size=8).astype(np.float32)
    gen = np.clip(pca_g.decode(Latent(array=z_s, nature="continuous")), 0, 1)

    for col_j in range(8):
        fig.add_trace(
            go.Heatmap(z=gen[col_j].reshape(28, 28)[::-1], colorscale="gray", showscale=False),
            row=row_i + 1, col=col_j + 1
        )
        fig.update_xaxes(visible=False, row=row_i + 1, col=col_j + 1)
        fig.update_yaxes(visible=False, row=row_i + 1, col=col_j + 1)

fig.update_layout(height=130 * len(k_values_gen), width=900,
                  title_text="Qualité de génération vs nombre de composantes")
fig.show()

**Conclusion (3)** : *A compléter après exécution.* Documenter rigoureusement les limites de la PCA comme générateur : images floues, artefacts de la linéarité, échec de l'hypothèse gaussienne en haute dimension. Ce résultat négatif est intéressant en soi et justifie le recours à des modèles non linéaires (AutoEncoder).

## 4. Sauvegarde des résultats (pour la synthèse comparative)

In [ ]:
pca_rd = {
    "method": "PCA",
    "points": [
        {
            "label": str(r["n_components"]),
            "total_bytes": int(r["total_compressed_bytes"]),
            "mse": float(r["reconstruction_mse"]),
            "codebook_bytes": int(r["codebook_bytes"]),
            "latent_bytes": int(r["latent_bytes"]),
        }
        for r in results
    ],
}

os.makedirs("data", exist_ok=True)
with open("data/pca_rate_distortion.json", "w") as f:
    json.dump(pca_rd, f, indent=2)

print("Résultats sauvegardés dans data/pca_rate_distortion.json")